In [14]:
# import packages 
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

In [15]:
# LOAD YOUR DATA HERE
# Replace 'your_file_name.csv' with the actual name of your outage dataset file
df = pd.read_csv('../data/cleaned_data.csv') 

# 1. Convert the column to actual datetime objects
df['start_time'] = pd.to_datetime(df['start_time'])

# 2. Extract the hour (0-23) into its own column
df['hour'] = df['start_time'].dt.hour

In [16]:
# 1. Setup Features and Target
# We use 'mean_customers', 'hour', and categorical dummies for 'state' and 'Event Type'
X = pd.get_dummies(df[['state', 'Event Type', 'mean_customers', 'hour']], drop_first=True)
y = df['duration']

# 2. Split and Scale (Using your preferred structure)
X_train_unscaled, X_test_unscaled, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_unscaled), columns=X.columns, index=X_train_unscaled.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_unscaled), columns=X.columns, index=X_test_unscaled.index)

# 3. Initialize Regression Models
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

# 4. Training and Metrics Loop
for name, model in models.items():
    # Cross-validation using R^2
    cv_r2 = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='r2').mean()
    
    # Fit on training data
    model.fit(X_train_scaled, y_train)
    
    # Predict on test set
    y_pred = model.predict(X_test_scaled)
    
    # Calculate Metrics
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    results.append({
        "Model": name, 
        "CV R2": cv_r2, 
        "Test R2": r2, 
        "MAE (Hours)": mae, 
        "RMSE (Hours)": rmse
    })

# 5. Compare Results
results_df = pd.DataFrame(results)
print(results_df)

               Model     CV R2   Test R2  MAE (Hours)  RMSE (Hours)
0  Linear Regression  0.277837  0.117657     6.482758     14.726674
1      Random Forest  0.943218  0.949290     0.960388      3.530463
2  Gradient Boosting  0.637799  0.640346     4.741815      9.402171


In [17]:
from sklearn.model_selection import GridSearchCV

# Parameter grid
lr_param_grid = {
    'fit_intercept': [True, False],
    'positive': [True, False]
}

# Initialize model
lr = LinearRegression()

# Grid Search
lr_grid = GridSearchCV(
    lr,
    lr_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

# Fit
lr_grid.fit(X_train_scaled, y_train)

# Best model
best_lr = lr_grid.best_estimator_

# Predictions
y_pred_lr = best_lr.predict(X_test_scaled)

# Metrics
print("Best Parameters:", lr_grid.best_params_)
print("Test R2:", r2_score(y_test, y_pred_lr))
print("MAE:", mean_absolute_error(y_test, y_pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))

Best Parameters: {'fit_intercept': True, 'positive': False}
Test R2: 0.11765701329735856
MAE: 6.482758125899894
RMSE: 14.726673780344296


In [ ]:
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf = RandomForestRegressor(random_state=42)

rf_grid = GridSearchCV(
    rf,
    rf_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

rf_grid.fit(X_train_scaled, y_train)

best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test_scaled)

print("Best Parameters:", rf_grid.best_params_)
print("Test R2:", r2_score(y_test, y_pred_rf))
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [ ]:
gb_param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0]
}

gb = GradientBoostingRegressor(random_state=42)

gb_grid = GridSearchCV(
    gb,
    gb_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

gb_grid.fit(X_train_scaled, y_train)

best_gb = gb_grid.best_estimator_
y_pred_gb = best_gb.predict(X_test_scaled)

print("Best Parameters:", gb_grid.best_params_)
print("Test R2:", r2_score(y_test, y_pred_gb))
print("MAE:", mean_absolute_error(y_test, y_pred_gb))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_gb)))